## Setup
Set `REPO_URL` to your GitHub repo, **or** leave it empty and attach the repo (uploaded as a Kaggle dataset) as an input.
Accelerator: **GPU T4 x2**. Internet: **On**.

In [ ]:
import os, glob, shutil, subprocess, json, time
REPO_URL = "https://github.com/areejfatima2998-creator/small-model-alignment.git"
WORK = "/kaggle/working"
REPO = f"{WORK}/small-model-alignment"
if not os.path.exists(REPO):
    if REPO_URL:
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO], check=True)
    else:
        src = [p for p in glob.glob("/kaggle/input/**/submission_adapter.py", recursive=True) if "/outputs/" not in p]
        assert src, "Attach the repo as a Kaggle dataset or set REPO_URL"
        shutil.copytree(os.path.dirname(src[0]), REPO, ignore=shutil.ignore_patterns("outputs"))
os.chdir(REPO)
os.environ["PYTHONPATH"] = REPO
os.environ["TOKENIZERS_PARALLELISM"] = "false"
print(os.getcwd())

In [ ]:
!python -m pip uninstall -y -q torchao || true
!python -m pip install -q "transformers>=4.43.0,<5" "datasets>=2.18,<5" huggingface_hub accelerate tqdm pyyaml sentencepiece safetensors pytest matplotlib
!python -m pip install -q -e . --no-deps

In [ ]:
!nvidia-smi
import torch, transformers, datasets
print("torch", torch.__version__, "| transformers", transformers.__version__, "| datasets", datasets.__version__)
print("GPUs:", torch.cuda.device_count(), [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

In [ ]:
!python -m pytest -q tests/

## Task 5: "Idiot Sandwich" - TRL DPOTrainer + LoRA on SmolLM2-360M-Instruct
Independent of the other notebooks. Expected: ~15-30 min.

In [ ]:
!python -m pip install -q "peft>=0.11.0" "trl>=0.12"
import trl, peft; print("trl", trl.__version__, "peft", peft.__version__)

In [ ]:
!python scripts/train_lora_dpo.py --output_dir outputs/lora_dpo

In [ ]:
job = subprocess.Popen("python scripts/generate.py --model_path HuggingFaceTB/SmolLM2-360M-Instruct --eval_set eval_sets/persona_style_prompts.json --generator smollm2-360m-instruct --output outputs/generations/persona_base.json --device cuda:0 --prompt_format chat --max_new_tokens 256 --batch_size 8 --repetition_penalty 1.1", shell=True, stdout=open("outputs/persona_base.log", "w"), stderr=subprocess.STDOUT)
!python scripts/generate.py --model_path HuggingFaceTB/SmolLM2-360M-Instruct --adapter_path outputs/lora_dpo/adapter --eval_set eval_sets/persona_style_prompts.json --generator smollm2-360m-lora-dpo --output outputs/generations/persona_lora.json --device cuda:1 --prompt_format chat --max_new_tokens 256 --batch_size 8 --repetition_penalty 1.1
job.wait(); assert job.returncode == 0, f"background job failed with exit code {job.returncode}"
!tail -3 outputs/persona_base.log
!python scripts/build_pairs.py --baseline outputs/generations/persona_base.json --candidate outputs/generations/persona_lora.json --output outputs/eval/lora_vs_base_pairs.jsonl

In [ ]:
for r_b, r_l in zip(json.load(open("outputs/generations/persona_base.json")), json.load(open("outputs/generations/persona_lora.json"))):
    print("=" * 80, "\n", r_b["instruction"], "\n--- BASE ---\n", r_b["output"][:600], "\n--- LORA ---\n", r_l["output"][:600])

In [ ]:
!cd outputs && zip -qr ../lora_run_outputs.zip . && cd .. && ls -la outputs/*